In [ ]:
# Running on Python version 3.13.7 - Created 4/13/2026

In [ ]:
'''
*Include variable name key*
*Implement code units?*
'''

import numpy as np

G = 6.674e-8
M_sun = 1.989e33

class Filament(x, mu, width):
    '''
    *Class documentation*
    '''

    def __init__(self, x, mu, width):
        self.mu = mu
        self.x = x
        self.width = width

        self.v_k = np.sqrt(G * M_sun / (x**3))
        self.v = drift_velocity(mu, self.v_k)
    
    def update_position(self, x):
        self.x = x
        self.v_k = keplarian_velocity(self.x)
        self.v = drift_velocity(self.x, self.mu)
    
    def get_x(self):
        return self.x
    
    def get_v(self):
        return self.v

    def get_width(self):
        return self.width
    
    def get_mu(self):
        return self.mu


def keplarian_velocity(x):
    '''
    Helper function, computes keplarian velocity
    '''
    return np.sqrt(G * M_sun / (x**3))


def drift_velocity(x, mu, eta = 1e-3, st = 1e-1): # Placeholder values for eta and st
    '''
    Computes drift velocity

    ---Params---
        mu: Dust-to-gas ratio
        eta: Gas pressure support parameter?
        st: Stoke's Number

    ---Returns---
        Drift velocity
    '''

    v_k = keplarian_velocity(x)

    return -(v_k * (1 + mu) * 2 * eta * st) / ((1 + mu)**2 + st**2)


def merge_filaments(filaments):
    '''
    Checks if merge conditions have been met for filaments and merges them, returns updated Numpy array of filament objects

    --Params--
        filaments: Numpy array of filament objects

    --Returns--
        filaments_new: updated list of filament objects
    '''

    regions = []
    filaments_new = np.copy(filaments)

    for filament in filaments:
        x, width = filament.get_x(), filament.get_width()
        regions.append((x-width, x+width))
    
    for i in range(len(regions)):
        for j in range(len(regions)):
            if(j > i):
                if(regions[i][1] >= regions[j][0]): # Checks overlapping widths
                    filaments_new[i] = merge([filaments_new[i], filaments_new[j]])
                    np.delete(filaments_new, j)
    
    return filaments_new
    

def merge(filaments):
    '''
    Helper function. Creates new filament object provided merge conditions are met
    '''

    positions, mus, widths = [], [], []
    for filament in filaments:
        positions.append(filament.get_x())
        mus.append(filament.get_mu())
        widths.append(filament.get_width())
    
    x_new = np.average(positions)
    mu_new = np.sum(mus)
    width_new = np.sum(widths)

    return Filament(x_new, mu_new, width_new)
    

def rk4_step(filaments, dt):
    '''
    Implements RK4 time integration scheme

    --Params--
        filaments: Numpy array of filament objects

    --Returns--
        filaments_new : Updated array of filaments
    '''

    filaments_new = np.copy(filaments)

    for filament in filaments_new:
        x, mu = filament.get_x(), filament.get_mu()
        k1 = filament.v
        k2 = drift_velocity((x + .5*k1[-1]*dt), mu)
        k3 = drift_velocity((x + .5*k2[-1]*dt), mu)
        k4 = drift_velocity((x + k3[-1]*dt), mu)

        x_new = x + dt*(k1 + 2*k2 + 2*k3 + k4)/6
        filament.update_position(x_new)
    
    return filaments_new

    
def init_filament(params):
    ''' 
    Randomly initializes a filament using specified parameters, returns a Numpy array of filaments sorted by position
    '''


def run_sim(filaments, dt, steps):
    ''' 
    Runs filament merger simulation, records data at every time step

    --Params--
        filaments: array of filament objects

    --Returns--
        data: array of tuples recording t, n filaments, positions, dust-to-gas ratios
    '''

    data = [] # append tuple (dt, #filaments, positions, dust-to-gas ratios)
    filament_samp = np.copy(filaments)
    t = 0

    for _ in range(steps):
        filament_samp = rk4_step(filament_samp, dt)
        n = len(filament_samp)

        # Attempt to merge filaments
        filament_samp = np.copy(merge_filaments(filament_samp))
        n_new = len(filament_samp)

        # Continue to merge filaments to exhaust merge events
        if(n_new < n):
            merged = True
            n = n_new

            while(merged):
                filament_samp = np.copy(merge_filaments(filament_samp))
                n_new = len(filament_samp)
                
                if(n == n_new):
                    merged = False
        
        # Collect positions and dust-to-gas ratios
        positions, mus = [], []
        for filament in filament_samp:
            positions.append(filament.get_x())
            mus.append(filament.get_mu())

        data.append(t, n_new, positions, mus)
        t += dt
    
    return data

In [ ]:
# Initialize Filaments

# Trial run